<a href="https://colab.research.google.com/github/sila85/production_ml_demo/blob/main/ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow scikit-learn matplotlib seaborn numpy

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

In [ ]:
(x_train, y_train), (x_test, y_test) = (
    keras.datasets.cifar10.load_data()
)

class_names = [
    'Airplane', 'Automobile', 'Bird', 'Cat',
    'Deer', 'Dog', 'Frog', 'Horse',
    'Ship', 'Truck'
]

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)

print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(20):
    plt.subplot(4, 5, i + 1)

    plt.imshow(x_train[i])

    label = y_train[i][0]

    plt.title(class_names[label])
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
unique, counts = np.unique(
    y_train,
    return_counts=True
)

plt.figure(figsize=(10, 5))

plt.bar(
    [class_names[i] for i in unique],
    counts
)

plt.title("CIFAR-10 Class Distribution")
plt.xlabel("Classes")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)

plt.show()

In [ ]:
# Normalize images
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Create validation dataset
x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

print("Training:", x_train.shape)
print("Validation:", x_val.shape)
print("Testing:", x_test.shape)

In [ ]:
class Patches(layers.Layer):

    def __init__(self, patch_size):
        super().__init__()

        self.patch_size = patch_size

    def call(self, images):

        patches = tf.image.extract_patches(
            images=images,
            sizes=[
                1,
                self.patch_size,
                self.patch_size,
                1
            ],
            strides=[
                1,
                self.patch_size,
                self.patch_size,
                1
            ],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )

        batch_size = tf.shape(patches)[0]

        patch_dims = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [batch_size, -1, patch_dims]
        )

        return patches

In [ ]:
patch_size = 4

sample_image = tf.expand_dims(
    x_train[0],
    axis=0
)

patch_layer = Patches(patch_size)

sample_patches = patch_layer(sample_image)

print("Patch shape:", sample_patches.shape)

In [ ]:
plt.figure(figsize=(8, 8))

image = x_train[0]

for i, patch in enumerate(
    tf.reshape(sample_patches, (64, 4, 4, 3))
):

    plt.subplot(8, 8, i + 1)

    plt.imshow(patch.numpy())

    plt.axis('off')

plt.suptitle("Image Divided into 64 Patches")
plt.show()

In [ ]:
class PatchEncoder(layers.Layer):

    def __init__(
        self,
        num_patches,
        projection_dim
    ):

        super().__init__()

        self.num_patches = num_patches

        self.projection = layers.Dense(
            units=projection_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patches):

        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1
        )

        encoded = (
            self.projection(patches)
            + self.position_embedding(positions)
        )

        return encoded

In [ ]:
class ClassToken(layers.Layer):

    def build(self, input_shape):

        self.cls_token = self.add_weight(
            name="cls_token",
            shape=(1, 1, input_shape[-1]),
            initializer="random_normal",
            trainable=True
        )

    def call(self, inputs):

        batch_size = tf.shape(inputs)[0]

        cls_tokens = tf.tile(
            self.cls_token,
            [batch_size, 1, 1]
        )

        return tf.concat(
            [cls_tokens, inputs],
            axis=1
        )

In [ ]:
def transformer_block(
    inputs,
    projection_dim,
    num_heads,
    mlp_dim,
    dropout_rate=0.1
):

    # First normalization
    x1 = layers.LayerNormalization(
        epsilon=1e-6
    )(inputs)

    # Multi-head self-attention
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=projection_dim // num_heads,
        dropout=dropout_rate
    )(
        x1,
        x1
    )

    # Residual connection
    x2 = layers.Add()([
        inputs,
        attention_output
    ])

    # Second normalization
    x3 = layers.LayerNormalization(
        epsilon=1e-6
    )(x2)

    # Feed-forward network
    mlp_output = layers.Dense(
        mlp_dim,
        activation='gelu'
    )(x3)

    mlp_output = layers.Dropout(
        dropout_rate
    )(mlp_output)

    mlp_output = layers.Dense(
        projection_dim
    )(mlp_output)

    mlp_output = layers.Dropout(
        dropout_rate
    )(mlp_output)

    # Residual connection
    outputs = layers.Add()([
        x2,
        mlp_output
    ])

    return outputs

In [ ]:
def build_vit():

    input_shape = (32, 32, 3)

    patch_size = 4

    num_patches = (
        input_shape[0] // patch_size
    ) ** 2

    projection_dim = 128

    num_heads = 4

    mlp_dim = 256

    num_transformer_blocks = 4

    inputs = layers.Input(
        shape=input_shape
    )

    # Data augmentation
    x = layers.RandomFlip(
        "horizontal"
    )(inputs)

    x = layers.RandomRotation(
        0.05
    )(x)

    x = layers.RandomZoom(
        0.1
    )(x)

    # Extract image patches
    patches = Patches(
        patch_size
    )(x)

    # Patch embedding and positional encoding
    encoded_patches = PatchEncoder(
        num_patches,
        projection_dim
    )(patches)

    # Add classification token
    x = ClassToken()(
        encoded_patches
    )

    # Transformer encoder blocks
    for _ in range(num_transformer_blocks):

        x = transformer_block(
            x,
            projection_dim,
            num_heads,
            mlp_dim
        )

    # Final normalization
    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    # Extract classification token
    x = layers.Lambda(
        lambda z: z[:, 0]
    )(x)

    # Classification head
    x = layers.Dense(
        128,
        activation='gelu'
    )(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        10,
        activation='softmax'
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="Vision_Transformer"
    )

    return model

In [ ]:
vit_model = build_vit()

vit_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']
)

vit_model.summary()

In [ ]:
callbacks = [

    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),

    keras.callbacks.ModelCheckpoint(
        'best_vit_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

history_vit = vit_model.fit(

    x_train,
    y_train,

    validation_data=(
        x_val,
        y_val
    ),

    epochs=20,

    batch_size=128,

    callbacks=callbacks
)

In [ ]:
def build_cnn():

    inputs = layers.Input(
        shape=(32, 32, 3)
    )

    # Data augmentation
    x = layers.RandomFlip(
        "horizontal"
    )(inputs)

    x = layers.RandomRotation(
        0.05
    )(x)

    # First convolution block
    x = layers.Conv2D(
        32,
        (3, 3),
        activation='relu',
        padding='same'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(
        (2, 2)
    )(x)

    # Second convolution block
    x = layers.Conv2D(
        64,
        (3, 3),
        activation='relu',
        padding='same'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(
        (2, 2)
    )(x)

    # Third convolution block
    x = layers.Conv2D(
        128,
        (3, 3),
        activation='relu',
        padding='same'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(
        (2, 2)
    )(x)

    # Classification layers
    x = layers.Flatten()(x)

    x = layers.Dense(
        256,
        activation='relu'
    )(x)

    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(
        10,
        activation='softmax'
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name="Baseline_CNN"
    )

In [ ]:
cnn_model = build_cnn()

cnn_model.compile(

    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
cnn_callbacks = [

    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),

    keras.callbacks.ModelCheckpoint(
        'best_cnn_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

history_cnn = cnn_model.fit(

    x_train,
    y_train,

    validation_data=(
        x_val,
        y_val
    ),

    epochs=20,

    batch_size=128,

    callbacks=cnn_callbacks
)

In [ ]:
# Predictions
vit_predictions = vit_model.predict(
    x_test
)

vit_predicted_classes = np.argmax(
    vit_predictions,
    axis=1
)

y_true = y_test.flatten()

# Metrics
vit_accuracy = accuracy_score(
    y_true,
    vit_predicted_classes
)

vit_precision = precision_score(
    y_true,
    vit_predicted_classes,
    average='macro'
)

vit_recall = recall_score(
    y_true,
    vit_predicted_classes,
    average='macro'
)

vit_f1 = f1_score(
    y_true,
    vit_predicted_classes,
    average='macro'
)

print("VISION TRANSFORMER RESULTS")

print("Accuracy:", vit_accuracy)

print("Precision:", vit_precision)

print("Recall:", vit_recall)

print("F1-score:", vit_f1)

print("\nClassification Report")

print(
    classification_report(
        y_true,
        vit_predicted_classes,
        target_names=class_names
    )
)

In [ ]:
cnn_predictions = cnn_model.predict(
    x_test
)

cnn_predicted_classes = np.argmax(
    cnn_predictions,
    axis=1
)

cnn_accuracy = accuracy_score(
    y_true,
    cnn_predicted_classes
)

cnn_precision = precision_score(
    y_true,
    cnn_predicted_classes,
    average='macro'
)

cnn_recall = recall_score(
    y_true,
    cnn_predicted_classes,
    average='macro'
)

cnn_f1 = f1_score(
    y_true,
    cnn_predicted_classes,
    average='macro'
)

print("BASELINE CNN RESULTS")

print("Accuracy:", cnn_accuracy)

print("Precision:", cnn_precision)

print("Recall:", cnn_recall)

print("F1-score:", cnn_f1)

print("\nClassification Report")

print(
    classification_report(
        y_true,
        cnn_predicted_classes,
        target_names=class_names
    )
)

In [ ]:
results = {

    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1-score'
    ],

    'Vision Transformer': [
        vit_accuracy,
        vit_precision,
        vit_recall,
        vit_f1
    ],

    'CNN': [
        cnn_accuracy,
        cnn_precision,
        cnn_recall,
        cnn_f1
    ]
}

import pandas as pd

results_df = pd.DataFrame(results)

print(results_df)

In [ ]:
results_df.set_index(
    'Metric'
).plot(
    kind='bar',
    figsize=(10, 6),
    ylim=(0, 1)
)

plt.title(
    "Performance Comparison: ViT vs CNN"
)

plt.ylabel("Score")

plt.xlabel("Evaluation Metrics")

plt.xticks(rotation=0)

plt.legend(
    loc='lower right'
)

plt.grid(axis='y')

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    history_vit.history['accuracy'],
    label='ViT Training Accuracy'
)

plt.plot(
    history_vit.history['val_accuracy'],
    label='ViT Validation Accuracy'
)

plt.plot(
    history_cnn.history['accuracy'],
    label='CNN Training Accuracy'
)

plt.plot(
    history_cnn.history['val_accuracy'],
    label='CNN Validation Accuracy'
)

plt.title(
    "Training and Validation Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    history_vit.history['loss'],
    label='ViT Training Loss'
)

plt.plot(
    history_vit.history['val_loss'],
    label='ViT Validation Loss'
)

plt.plot(
    history_cnn.history['loss'],
    label='CNN Training Loss'
)

plt.plot(
    history_cnn.history['val_loss'],
    label='CNN Validation Loss'
)

plt.title(
    "Training and Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid()

plt.show()

In [ ]:
vit_cm = confusion_matrix(
    y_true,
    vit_predicted_classes
)

plt.figure(figsize=(11, 8))

sns.heatmap(
    vit_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title(
    "Vision Transformer Confusion Matrix"
)

plt.xlabel("Predicted Class")

plt.ylabel("Actual Class")

plt.show()

In [ ]:
cnn_cm = confusion_matrix(
    y_true,
    cnn_predicted_classes
)

plt.figure(figsize=(11, 8))

sns.heatmap(
    cnn_cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title(
    "CNN Confusion Matrix"
)

plt.xlabel("Predicted Class")

plt.ylabel("Actual Class")

plt.show()

In [ ]:
plt.figure(figsize=(14, 10))

for i in range(20):

    plt.subplot(4, 5, i + 1)

    plt.imshow(x_test[i])

    actual = class_names[y_true[i]]

    predicted = class_names[
        vit_predicted_classes[i]
    ]

    color = (
        'green'
        if actual == predicted
        else 'red'
    )

    plt.title(
        f"Actual: {actual}\nPredicted: {predicted}",
        color=color,
        fontsize=9
    )

    plt.axis('off')

plt.suptitle(
    "Vision Transformer Classification Results"
)

plt.tight_layout()

plt.show()